# Benchmark de obstrução: ViDDAR com diferentes VLMs

Este notebook roda a detecção de **ataques de obstrução** do ViDDAR (Xiu et al., IEEE TVCG 2025, §3.1,
§4.1 e §5.1) com os mesmos VLMs do benchmark de manipulação, sem mudar nada no pipeline replicado. A
pergunta é **como a escolha do VLM afeta o detector** e, em especial, **de onde vêm os falsos
positivos**.

**A tarefa.** Cada amostra é um par (imagem crua I_r, imagem aumentada I_a). Há obstrução quando o
conteúdo virtual cobre um objeto real importante, o *objeto-chave* (placa, extintor, tomada...). São 306
amostras de 23 classes de objeto, 145 obstruídas (rótulo 1) e 161 não obstruídas (rótulo 0).

**O pipeline do ViDDAR** (o mesmo da replicação, com os prompts literais do artigo):
1. o VLM recebe I_r e nomeia o objeto-chave;
2. o Grounding DINO localiza esse nome em I_r e o SAM recorta a máscara do objeto, m_k;
3. a máscara do conteúdo virtual, m_c, vem da diferença de pixels entre I_r e I_a;
4. há obstrução se |m_k ∩ m_c| / |m_k| ≥ α, com α = 0,25 (Eq. 4).

O VLM só participa do passo 1. O resto é geometria, e é isso que o artigo argumenta: separar o
reconhecimento (VLM) da decisão (geometria) é mais confiável que perguntar ao VLM se há obstrução.

## Métodos (Tabela 1 do artigo)

| Método | O que faz | Usa VLM |
|---|---|---|
| `viddar` | o pipeline acima | sim, só no passo 1 |
| `end_to_end` | o VLM nomeia o objeto e decide sozinho se ele está obstruído (vê I_r e I_a) | sim, decide tudo |
| `underdetailed` | como o ViDDAR, com um prompt mínimo para o objeto-chave | sim |
| `greedy` | o VLM lista vários objetos; há obstrução se qualquer um passar de α | sim |
| `prior` | o detector recebe o **nome verdadeiro** do objeto: limite superior do pipeline | não |
| `saliency`, `canny` | linhas de base clássicas de visão computacional | não |

Os três métodos sem VLM não dependem do modelo: rodam uma vez e servem de referência.

## Roteiro

| Seção | O que mostra |
|---|---|
| 2 | configuração |
| 3 | execução (só roda o que ainda não existe em `results/`) |
| **4** | **resultado geral por modelo e método (Tabela 1) e ViDDAR × end-to-end** |
| **5** | **análise dos erros: em que etapa do pipeline cada FP e FN acontece** |
| 6 | sensibilidade ao limiar α |
| 7 | interpretação e resumo |

## Convenções

- **Reconhecimento do objeto-chave**, dois critérios:
  - **léxico:** a frase do VLM nomeia a classe certa (permissivo: "black tv stand" conta como TV);
  - **com máscara (IoU ≥ 0,5):** a máscara recortada a partir da frase cobre o objeto verdadeiro. É o
    critério mais confiável para conclusões.
- **mIoU:** IoU médio entre a máscara do objeto prevista e a verdadeira.
- **Acurácia** = acertos / 306. **Precisão** = obstruções reais entre as amostras marcadas.
  **Recall** = obstruções detectadas / 145. **FPR** = não obstruídas marcadas / 161. **FNR** = 1 − recall.
- Resposta sem sim/não (só no `end_to_end`) conta como erro.
- **McNemar:** teste pareado sobre as mesmas 306 amostras; uma diferença só conta com p < 0,05.

## Antes de começar

- Use o kernel **"Python (ViDDAR .venv)"** e rode as células em ordem.
- A lógica fica em `viddar/`; a mesma execução roda pela linha de comando com
  `scripts/run_obstruction.py --methods viddar end_to_end underdetailed greedy --vlm <modelo>`.
- As respostas do VLM ficam em cache (`results/cache/`): execuções prontas só são relidas.

## 1. Ambiente

In [ ]:
import os
import sys
from pathlib import Path

# O notebook pode ser aberto a partir da raiz ou da pasta notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "viddar").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from viddar import obstruction, reporting, viz
from viddar.experiments import ObstructionConfig, load_run, run_obstruction

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 250)
print("raiz do repositório:", ROOT)

## 2. Configuração

- **`MODELS`:** os mesmos presets do benchmark de manipulação (`viddar/vlm.py`).
- **`METHODS`:** métodos com VLM, rodados para cada modelo. **`BASELINES`:** métodos sem VLM, rodados
  uma vez (pasta `results/obstruction/no_vlm/`).
- **`ALPHA`:** limiar da Eq. 4 (artigo: 0,25).
- **`RUN_MISSING`:** com `True`, executa o que ainda não existe; com `False`, só lê `results/`.
- **`LIMIT`:** número de amostras, para testes rápidos. Grava em `results/_smoke`; apague a pasta depois.

In [ ]:
MODELS = ["qwen3-vl-2b", "qwen3-vl-4b", "qwen3-vl-8b-4bit", "llava-next-8b-4bit", "gemma4-12b-4bit"]
METHODS = ["viddar", "end_to_end", "underdetailed", "greedy"]
BASELINES = ["prior", "saliency", "canny"]
ALPHA = 0.25

RUN_MISSING = True   # executa o que ainda não existe
LIMIT = None         # None = dataset completo; ex.: 5 para teste rápido

MANIFEST = "data/manifests/viddar_obstruction.csv"
OUT_ROOT = Path("results" if LIMIT is None else "results/_smoke")
print("resultados em:", OUT_ROOT)

## 3. Execução

Para cada modelo, roda os métodos de `METHODS` que ainda não têm `metrics.json`, em estágios: primeiro
todas as chamadas ao VLM, depois o modelo é descarregado e entram o Grounding DINO e o SAM (um modelo
por vez na GPU). Os métodos sem VLM rodam uma vez. Com o dataset completo, cada modelo novo leva cerca de
1 a 3 h, conforme o tamanho; o `viddar` e o `end_to_end` compartilham a pergunta do objeto-chave pelo
cache.

In [ ]:
def missing(model_dir: str, methods: list[str]) -> list[str]:
    return [m for m in methods if not (OUT_ROOT / "obstruction" / model_dir / m / "metrics.json").exists()]


todo = {model: missing(model, METHODS) for model in MODELS}
todo["no_vlm"] = missing("no_vlm", BASELINES)
for model, methods in todo.items():
    if not methods:
        continue
    if not RUN_MISSING:
        print("falta:", model, methods)
        continue
    print("executando:", model, methods)
    run_obstruction(ObstructionConfig(manifest=MANIFEST, methods=methods,
                                      vlm=MODELS[0] if model == "no_vlm" else model, alpha=ALPHA,
                                      limit=LIMIT, out=str(OUT_ROOT / "obstruction")))

runs = reporting.load_obstruction_runs(OUT_ROOT, MODELS, METHODS, BASELINES)
print(f"{len(runs)} execuções carregadas:", sorted(f"{m}/{k}" for m, k in runs))

## 4. Resultado geral (Tabela 1 do artigo)

Uma linha por modelo e método, com as métricas da Tabela 1 (reconhecimento do objeto-chave, mIoU e
acurácia) e também precisão, recall, FPR e FNR. As linhas `no_vlm` são as referências sem VLM. Os
números do artigo vêm logo abaixo.

In [ ]:
bench = reporting.obstruction_benchmark(runs)
order = {m: i for i, m in enumerate(MODELS + [reporting.NO_VLM])}
bench = bench.sort_values(["método", "modelo"], key=lambda s: s.map(order) if s.name == "modelo"
                          else s.map({m: i for i, m in enumerate(METHODS + BASELINES)}))
display(bench.reset_index(drop=True))
display(Markdown("**Referência: Tabela 1 do artigo** (em %)"))
display(reporting.PAPER_TABLE1)

### 4.1 ViDDAR × end-to-end

A comparação central do artigo: deixar a decisão com a geometria (ViDDAR) é melhor que perguntar ao
VLM se o objeto está obstruído (end-to-end)? O teste de McNemar compara os dois nas mesmas 306 amostras.

In [ ]:
display(reporting.obstruction_vs(runs, MODELS, "viddar", "end_to_end"))

## 5. Análise dos erros do ViDDAR

### 5.1 Em que etapa cada erro acontece

Cada erro do ViDDAR recebe a etapa do pipeline que falhou, nesta ordem:

1. **nada detectado:** o Grounding DINO não encontrou o objeto que o VLM nomeou. A razão fica 0, então
   só gera FN.
2. **objeto errado:** a máscara recortada não é o objeto-chave (IoU < 0,5 com a máscara verdadeira).
   Quase sempre o VLM nomeou outro objeto; às vezes o detector escolheu outro objeto com o mesmo nome.
3. **objeto certo, razão do lado errado de α:** o objeto certo foi recortado, mas a fração coberta pelo
   conteúdo ficou do outro lado do limiar, por máscara imprecisa ou por um rótulo perto do limite.

Cada célula traz "erros / amostras possíveis" (161 não obstruídas para FP, 145 obstruídas para FN). Entre
parênteses, quantas dessas mesmas amostras o `prior` também erra. O `prior` recebe o nome verdadeiro do
objeto, então:

- se ele **acerta**, o erro é do VLM (reconhecimento);
- se ele **também erra**, o problema está depois do VLM (detector, SAM, α ou o próprio rótulo).

In [ ]:
display(reporting.obstruction_cause_comparison(runs, MODELS, "viddar"))
prior = runs.get((reporting.NO_VLM, "prior"))
if prior is not None:
    display(Markdown("**`prior` (nome verdadeiro do objeto), para comparação**"))
    display(obstruction.cause_table(prior))

### 5.2 Respostas mais frequentes do VLM

As cinco frases de objeto-chave mais comuns de cada modelo, com o número de ocorrências. O dataset tem
23 classes, então uma frase repetida dezenas de vezes que não corresponde às cenas indica que o modelo
copia o exemplo do prompt (o prompt do artigo cita "a red box").

In [ ]:
display(reporting.top_key_phrases(runs, MODELS, "viddar", n=5))

### 5.3 Por classe de objeto

FPR do ViDDAR (%) por classe e modelo. As colunas `n` e `obstruídos` dão o tamanho de cada classe: com
poucas amostras, um único erro muda muito a taxa.

In [ ]:
display(reporting.obstruction_by_group(runs, MODELS, "viddar", metric="fpr"))

### 5.4 Exemplos

Escolha um modelo e um tipo de erro. Verde: objeto-chave verdadeiro; laranja: conteúdo virtual. O
título mostra o que o VLM respondeu, a razão de obstrução e o IoU da máscara prevista.

In [ ]:
EX_MODEL, EX_ERROR, N_EXAMPLES = "qwen3-vl-4b", "FP", 2   # EX_ERROR: "FP" ou "FN"

run_dir = OUT_ROOT / "obstruction" / EX_MODEL / "viddar"
if (run_dir / "predictions.csv").exists():
    pred = load_run(run_dir, manifest=MANIFEST, kind="obstruction").predictions
    pred["cause"] = obstruction.error_causes(pred)
    sample = pred[pred["cause"].str.startswith(EX_ERROR)]
    print(f"{EX_MODEL}: {len(sample)} {EX_ERROR}", sample["cause"].value_counts().to_dict())
    for _, row in sample.head(N_EXAMPLES).iterrows():
        viz.show_obstruction(row)
        plt.show()

## 6. Sensibilidade ao limiar α

A razão |m_k ∩ m_c| / |m_k| fica salva por amostra, então a acurácia para qualquer α é recalculada sem
chamar o modelo. Uma linha por modelo (ViDDAR) e a do `prior` como referência. Se as curvas de todos os
modelos tiverem o pico no mesmo α, o limiar do artigo vale para qualquer VLM; a distância até a curva do
`prior` é o que se perde no reconhecimento do objeto.

In [ ]:
curves = {model: runs[(model, "viddar")] for model in MODELS if (model, "viddar") in runs}
if prior is not None:
    curves["prior (nome verdadeiro)"] = prior
if curves:
    fig, sweep = viz.alpha_sweep(curves, paper_alpha=ALPHA)
    plt.show()
    display((sweep.set_index("alpha") * 100).round(1))

## 7. Interpretação

*Escrita com os resultados do Qwen3-VL-2B, Qwen3-VL-4B e LLaVA-Next-8B; revisar quando os modelos novos
terminarem.*

- **A tese do artigo se mantém em todos os modelos.** O ViDDAR supera o end-to-end com folga (McNemar
  p < 0,001 nos três). Com o Qwen3-VL-4B, a acurácia do ViDDAR (91,5%) fica no nível do GPT-4o no
  artigo (91,8%).
- **Os falsos positivos do ViDDAR são todos de reconhecimento.** Em todos os modelos, cada FP é um caso
  de "objeto errado": o VLM nomeia outro objeto, que o conteúdo virtual cobre. Nenhum FP vem do limiar.
  O `prior`, que recebe o nome certo, tem só 1 FP em 161. Melhorar o detector passa por melhorar o
  reconhecimento do objeto-chave, não a geometria.
- **Os falsos negativos têm duas origens.** Parte é "objeto errado" (o VLM perdeu o objeto); a outra
  parte é "objeto certo, razão do lado errado de α", e nessas o `prior` também erra quase sempre: é um
  limite do limiar ou do rótulo, não do VLM.
- **No end-to-end, os modelos erram para lados opostos**, como na manipulação: o Qwen3-VL-2B diz
  "obstruído" quase sempre (FPR de 88%), e o LLaVA quase nunca (recall de 8%).
- **O Qwen3-VL-2B copia o exemplo do prompt:** "red box" é a frase mais frequente dele (36 vezes), e
  não existe essa classe no dataset.

### Resumo

Uma linha por modelo com o ViDDAR e o end-to-end, e a distância até o `prior` (o que se perde no
reconhecimento do objeto-chave). Antes de concluir, leia algumas respostas brutas dos modelos novos
(seção 5.2 e `predictions.csv`).

In [ ]:
rows = []
prior_acc = bench.loc[bench["método"] == "prior", "acurácia"]
prior_acc = float(prior_acc.iloc[0]) if len(prior_acc) else float("nan")
for model in MODELS:
    b = bench[bench["modelo"] == model].set_index("método")
    if "viddar" not in b.index:
        continue
    rows.append({"modelo": model,
                 "objeto-chave (IoU ≥ 0,5)": b.loc["viddar", "objeto-chave (IoU ≥ 0,5)"],
                 "ViDDAR: acurácia": b.loc["viddar", "acurácia"], "ViDDAR: FPR": b.loc["viddar", "FPR"],
                 "ViDDAR: FNR": b.loc["viddar", "FNR"],
                 "end-to-end: acurácia": b["acurácia"].get("end_to_end"),
                 "distância até o prior (p.p.)": round(prior_acc - b.loc["viddar", "acurácia"], 1)})
display(pd.DataFrame(rows))